<!-- project-navigation -->
# lime
Role: Model explanation.

See [notebook guide](README.md) and [model comparison](../reports/model_comparison.md). Historical code and outputs are retained; run with the project root as working directory.


In [2]:
import sys
!{sys.executable} -m pip install lime

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.0 MB 3.6 MB/s eta 0:00:04
   -- ------------------------------------- 0.8/12.0 MB 3.6 MB/s eta 0:00:04
   ------ --------------------------------- 1.8/12.0 MB 3.4 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.0 MB 3.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.0 MB 3.2 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/12.0 MB 2.9 MB/s eta 0:00:03
   ------------ --------------------------- 3.7/12.0 MB 3.0 MB/s eta 0:00:03
   ------------ --------------------------- 3.7/12.0 MB 3.0 M

In [3]:
from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

In [5]:
import joblib

model_lgb_spw = joblib.load(
    "data/models/lightgbm_scale_pos_weight.pkl"
)

print("Modèle chargé :", type(model_lgb_spw))

Modèle chargé : <class 'lightgbm.sklearn.LGBMClassifier'>


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

# ============================================================
# RECONSTRUCTION DES DONNÉES POUR LIME
# ============================================================

df = pd.read_csv("data/processed/train_encoded.csv")

print("Dimension :", df.shape)

X = df.drop(columns=["TARGET", "SK_ID_CURR"], errors="ignore")
y = df["TARGET"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\n========== DIMENSIONS ==========")
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)
print("y_train :", y_train.shape)
print("y_valid :", y_valid.shape)

Dimension : (307511, 242)

========== DIMENSIONS ==========
X_train : (246008, 240)
X_valid : (61503, 240)
y_train : (246008,)
y_valid : (61503,)


In [10]:
import joblib

model_lgb_spw = joblib.load(
    "data/models/lightgbm_scale_pos_weight.pkl"
)

print("Modèle chargé :", type(model_lgb_spw))

Modèle chargé : <class 'lightgbm.sklearn.LGBMClassifier'>


In [11]:
import numpy as np

X_lime_train = X_train.sample(
    n=5000,
    random_state=42
)

explainer_lime = LimeTabularExplainer(
    training_data=X_lime_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=["Non-défaut", "Défaut"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

print("✓ Explainer LIME créé")

✓ Explainer LIME créé


In [12]:
indices_defaut = np.where(y_valid.values == 1)[0]

idx = indices_defaut[0]

instance = X_valid.iloc[idx]

proba_defaut = model_lgb_spw.predict_proba(
    instance.values.reshape(1, -1)
)[0, 1]

print("Index client :", idx)
print("Vraie classe :", y_valid.iloc[idx])
print(f"Probabilité prédite de défaut : {proba_defaut:.4f}")

Index client : 9
Vraie classe : 1
Probabilité prédite de défaut : 0.5739


c:\Users\Asus\OneDrive\Bureau\FinTech\.venv-1\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
exp = explainer_lime.explain_instance(
    data_row=instance.values,
    predict_fn=model_lgb_spw.predict_proba,
    num_features=15
)

print("✓ Explication LIME générée")

c:\Users\Asus\OneDrive\Bureau\FinTech\.venv-1\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✓ Explication LIME générée


In [14]:
print("========== VARIABLES IMPORTANTES — LIME ==========")

for feature, weight in exp.as_list():
    print(f"{feature} : {weight:.4f}")

========== VARIABLES IMPORTANTES — LIME ==========
TOTAL_OVERDUE_AMOUNT <= 0.00 : -0.1573
EXT_SOURCE_3 <= 0.41 : 0.1563
EXT_SOURCE_1 <= 0.43 : 0.1131
HOUSETYPE_MODE_terraced house <= 0.00 : -0.0815
OCCUPATION_TYPE_Realty agents <= 0.00 : -0.0765
ORGANIZATION_TYPE_Military <= 0.00 : 0.0697
ORGANIZATION_TYPE_Religion <= 0.00 : 0.0696
ORGANIZATION_TYPE_Police <= 0.00 : 0.0685
AVG_CNT_INSTALMENT_FUTURE > 11.15 : 0.0670
CODE_GENDER <= 0.00 : -0.0634
OCCUPATION_TYPE_Low-skill Laborers <= 0.00 : -0.0546
ORGANIZATION_TYPE_Restaurant <= 0.00 : 0.0537
ORGANIZATION_TYPE_Mobile <= 0.00 : 0.0394
ORGANIZATION_TYPE_Trade: type 4 <= 0.00 : -0.0197
ORGANIZATION_TYPE_Cleaning <= 0.00 : -0.0181
